# Notebook 7b: Figures at print size

**Runtime:** CPU, about 1 minute. No GPU, no models.

**Why:** the figures in the paper were drawn about 1.5 times larger than they are printed, so after shrinking, the text was only 4–6 pt (Springer asks for about 8 pt). This notebook draws every figure at its **final printed width**, with 8 pt text, and without a title inside the figure (the caption carries the message).

The numbers are the same as in the paper: Figures 1 and 3 use the values printed in Table 1/Fig. 1 and Table 6; Figures 2, S1 and S2 read the same result files as Notebook 7.

At the end, a zip with all five figures downloads to your computer.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')

import os, re, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
OUT = f'{RESULTS}/paper/figures_v2'
os.makedirs(OUT, exist_ok=True)

plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8,
                     'ytick.labelsize': 8, 'legend.fontsize': 7.5, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.3,
                     'pdf.fonttype': 42, 'figure.dpi': 150})
ACCENT, GREY, DARK = '#2a6f97', '#9aa5b1', '#1f2933'
MAIN_W = 5.2     # main paper text width in inches (sn-jnl)
SUPP_W = 6.3     # supplementary text width in inches (A4, 2.5 cm margins)

def R(name):
    return pd.read_csv(f'{RESULTS}/{name}')
def ci(s):
    a = re.findall(r'-?\d+\.?\d*(?:e-?\d+)?', str(s))
    return (float(a[0]), float(a[1])) if len(a) >= 2 else (np.nan, np.nan)
def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/{name}.{ext}', bbox_inches='tight', pad_inches=0.02, dpi=300)
    w, h = fig.get_size_inches()
    plt.show()
    print(f'saved {name}  ({w:.2f} x {h:.2f} in)')

## Figure 1: label composition (printed at 0.8 × text width)

In [ ]:
order = ['only Potential', 'only Good', 'both Good and Potential', 'no fit candidate']
counts = {'only Potential': 162, 'only Good': 52, 'both Good and Potential': 91, 'no fit candidate': 46}
total = sum(counts.values()); assert total == 351
fig, ax = plt.subplots(figsize=(0.8 * MAIN_W, 1.45))
colors = [GREY, GREY, ACCENT, '#d9dee3']
vals = [counts[k] for k in order]
ax.barh(order[::-1], vals[::-1], color=colors[::-1])
for i, v in enumerate(vals[::-1]):
    ax.text(v + 3, i, f'{v} ({v / total:.0%})', va='center')
ax.set_xlabel('jobs'); ax.set_xlim(0, max(vals) * 1.3); ax.grid(axis='y', visible=False)
save(fig, 'fig1_label_composition')

## Figure 2: memorisation grid (printed at full text width)

In [ ]:
s6 = R('06a_cv_summary.csv').set_index('method')
show = ['FT standard', 'FT size-matched', 'FT strict', 'BGE-base (zero-shot)', 'E5-base (zero-shot)', 'TF-IDF']
fig, axes = plt.subplots(1, 2, figsize=(MAIN_W, 2.0), sharey=True)
for ax, col, title in [(axes[0], 'AUC fit-vs-no', 'Fit vs no fit'), (axes[1], 'AUC good-vs-potential', 'Good vs Potential')]:
    x = s6.loc[show]
    lo_hi = np.array([ci(v) for v in x[col + ' CI']])
    y = np.arange(len(show))[::-1]
    colors = [ACCENT if m.startswith('FT') else GREY for m in show]
    ax.errorbar(x[col], y, xerr=[x[col] - lo_hi[:, 0], lo_hi[:, 1] - x[col]], fmt='none', ecolor=DARK, elinewidth=0.9, capsize=2)
    ax.scatter(x[col], y, c=colors, zorder=3, s=20)
    ax.set_yticks(y); ax.set_yticklabels(show); ax.set_title(title, loc='left'); ax.set_xlabel('AUC (pooled)')
    if col == 'AUC good-vs-potential':
        ax.axvline(0.5, color=DARK, lw=0.8, ls=':')
print(x[['AUC fit-vs-no', 'AUC good-vs-potential']].round(3))   # must match Table 5
save(fig, 'fig2_memorisation')

## Figure 3: accuracy vs cost (printed at full text width)
Values from Table 6 of the paper. The five methods that sit together at 0.78–0.79 get one shared label instead of five overlapping ones.

In [ ]:
pts = [  # method, seconds per job, NDCG@10 (Table 6), is LLM
    ('Random', 0.0001, 0.644, False), ('TF-IDF', 0.084, 0.780, False), ('BM25', 0.119, 0.695, False),
    ('E5-base', 0.032, 0.784, False), ('BGE-base', 0.032, 0.781, False), ('MPNet-base', 0.028, 0.791, False),
    ('BGE-base (fine-tuned)', 0.016, 0.833, False), ('BGE-base (fine-tuned, strict)', 0.016, 0.791, False),
    ('Qwen3-4B judge', 27.4, 0.807, True), ('Qwen3-1.7B judge', 11.9, 0.759, True),
    ('E5 → Qwen3-4B (top 10)', 13.7, 0.780, True), ('E5 → Qwen3-1.7B (top 10)', 6.0, 0.763, True)]
LABEL = {  # offset in points, alignment
    'Random': (5, 0, 'left'), 'BM25': (5, 0, 'left'), 'BGE-base (fine-tuned)': (5, 0, 'left'),
    'Qwen3-4B judge': (5, 0, 'left'), 'E5 → Qwen3-4B (top 10)': (5, 0, 'left'),
    'E5 → Qwen3-1.7B (top 10)': (-4, -9, 'right'), 'Qwen3-1.7B judge': (5, -6, 'left')}
fig, ax = plt.subplots(figsize=(MAIN_W, 2.4))
for name, s, nd, llm in pts:
    ax.scatter(s, nd, color=ACCENT if llm else GREY, s=22, zorder=3)
    if name in LABEL:
        dx, dy, ha = LABEL[name]
        ax.annotate(name, (s, nd), xytext=(dx, dy), textcoords='offset points', ha=ha, va='center', fontsize=7.5)
ax.annotate('TF-IDF, E5, BGE, MPNet,\nfine-tuned BGE (strict)', xy=(0.014, 0.786), xytext=(1.2e-4, 0.733),
            ha='left', va='center', fontsize=7.5, arrowprops=dict(arrowstyle='-', color=GREY, lw=0.6))
ax.set_xscale('log'); ax.set_xlim(5e-5, 1500); ax.set_ylim(0.63, 0.85)
ax.set_xlabel('seconds to rank one job (log scale, T4 GPU)'); ax.set_ylabel('NDCG@10, test')
ax.scatter([], [], color=GREY, s=22, label='keyword / embedding'); ax.scatter([], [], color=ACCENT, s=22, label='LLM judge')
ax.legend(loc='lower right', frameon=False)
save(fig, 'fig3_accuracy_vs_cost')

## Figures S1 and S2 (Supplementary, printed at 0.8 × and full supplementary width)

In [ ]:
auc = R('05_auc_by_labels.csv'); gp = auc[auc.task == 'good vs potential']
METHODS = ['Resume-prior', 'TF-IDF', 'E5-base', 'BGE-base', 'MPNet-base', 'Qwen3-4B judge', 'Qwen3-1.7B judge']
fig, ax = plt.subplots(figsize=(0.8 * SUPP_W, 2.6))
y = np.arange(len(METHODS))[::-1]
for labels, color, dy in [('dataset', GREY, 0.18), ('you', ACCENT, -0.18)]:
    x = gp[gp.labels == labels].set_index('method').reindex(METHODS)
    lo_hi = np.array([ci(v) for v in x.CI])
    ax.errorbar(x.AUC, y + dy, xerr=[x.AUC - lo_hi[:, 0], lo_hi[:, 1] - x.AUC], fmt='o', color=color, ms=3.5,
                elinewidth=0.9, capsize=2, label='dataset labels' if labels == 'dataset' else 'human labels')
ax.axvline(0.5, color=DARK, lw=0.8, ls=':')
ax.set_yticks(y); ax.set_yticklabels(METHODS); ax.set_xlabel('AUC, Good vs Potential')
ax.legend(loc='lower left', bbox_to_anchor=(0, 1.0), ncol=2, frameon=False)
save(fig, 'fig4_good_vs_potential')

steps = R('08b_steps.csv'); extra = R('08c_markdown_steps.csv')
meths = ['tfidf', 'e5', 'bge', 'mpnet', 'qwen4b judge']
names = {'tfidf': 'TF-IDF', 'e5': 'E5', 'bge': 'BGE', 'mpnet': 'MPNet', 'qwen4b judge': 'Qwen3-4B judge'}
bars = [('AI polish, total', steps, 'total'), ('restoring spaces only', steps, 'cleaning'),
        ('markdown in the AI text', steps, 'markdown symbols'), ('markdown added to original', extra, 'all formatting')]
fig, ax = plt.subplots(figsize=(SUPP_W, 2.6))
w = 0.8 / len(bars); palette = [ACCENT, '#d9dee3', '#7fb3d5', '#e07a5f']
for k, (label, df, step) in enumerate(bars):
    x = df[df.step == step].set_index('method').reindex(meths)
    val = x['change in SDs'].to_numpy(dtype=float)
    lo_hi = np.array([ci(v) for v in x['CI (SDs)']])
    pos = np.arange(len(meths)) + (k - (len(bars) - 1) / 2) * w
    ax.bar(pos, val, w, color=palette[k], label=label)
    ok = ~np.isnan(val)
    ax.errorbar(pos[ok], val[ok], yerr=[val[ok] - lo_hi[ok, 0], lo_hi[ok, 1] - val[ok]], fmt='none', ecolor=DARK, elinewidth=0.8, capsize=1.5)
ax.axhline(0, color=DARK, lw=0.8)
ax.set_xticks(np.arange(len(meths))); ax.set_xticklabels([names[m] for m in meths])
ax.set_ylabel('score change (SDs)'); ax.legend(frameon=False)
save(fig, 'fig5_polish')

## Extra (reviewer point 8): how many "one-label" jobs have only one fit candidate?
A job with a single fit candidate is trivially "all Good" or "all Potential". We count them, and compare the number of one-label jobs with what we would expect if Good/Potential were shuffled at random across all fit pairs (each job keeps its number of fit candidates; 2,000 shuffles, seed 42).

In [ ]:
DATA = '/content/drive/MyDrive/resume-job-matching/data'
all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet') for s in ('train', 'val', 'test')], ignore_index=True)
fit = all_df[all_df.relevance > 0]
g = fit.groupby('job_id').relevance.agg(n='size', good=lambda r: int((r == 2).sum()))
g['uniform'] = (g.good == 0) | (g.good == g.n)
print(f'jobs with a fit candidate: {len(g)}; one-label jobs: {int(g.uniform.sum())} (paper: 305 and 214)')
print(f'one-label jobs with only ONE fit candidate: {int((g.uniform & (g.n == 1)).sum())}')
print(f'one-label jobs with 2+ fit candidates: {int((g.uniform & (g.n >= 2)).sum())}')
print('fit candidates per job, one-label jobs: median', g[g.uniform].n.median(), '| mixed jobs: median', g[~g.uniform].n.median())
rng = np.random.default_rng(42); labels = (fit.relevance == 2).to_numpy(); sizes = g.n.reindex(fit.job_id.unique()).to_numpy()
order = fit.job_id.to_numpy(); jobs = pd.unique(order); idx = {j: np.where(order == j)[0] for j in jobs}
null = []
for _ in range(2000):
    s = rng.permutation(labels); u = 0
    for j in jobs:
        v = s[idx[j]]; u += int(v.all() or (~v).all())
    null.append(u)
null = np.array(null)
print(f'one-label jobs expected under random shuffling: mean {null.mean():.1f}, 95% range [{np.percentile(null, 2.5):.0f}, {np.percentile(null, 97.5):.0f}]; '
      f'observed {int(g.uniform.sum())}; p = {(np.sum(null >= g.uniform.sum()) + 1) / (len(null) + 1):.4f}')

## Download
A zip with all five figures (PDF and PNG) downloads to your computer. It is also kept on Drive in `results/paper/figures_v2`.

In [ ]:
shutil.make_archive('/content/figures_v2', 'zip', OUT)
files.download('/content/figures_v2.zip')